In [171]:
import pandas as pd
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold
)
import json
import re


In [142]:
import import_ipynb
from get_zip_codes import get_zip_codes

Orders

In [143]:
orders = pd.read_excel('data/Shipped Order Data by Order and line level details for POC.xlsx')

In [144]:
orders

,ORDER_NO,ORDER_SUFFIX,ORDER_LINE_NO,ORDER_ENTERED_DATE,INVOICED_DATE,TRANS_TYPE,CUSTOMER_NO,SHIP_TO_NO,addr_1,addr_2,city,state,zipcd,JOB_TYPE,JOB_TYPE_DESCRIPTION,JOB_SUBTYPE,JOB_SUBTYPE_DESCRIPTION,WAREHOUSE,PRODUCT_CODE,descrip_1,PRODUCT_CAT_DESC,PRODUCT_PRIMARY_CAT_DESC,REPORT_CATEGORY,STOCK_QTY_SHIPPED,STOCK_UOM
0,10000434,0,1,2024-02-01,2024-02-01,so,93290,MISC,1528 W REMINGTON LN,NaN,ROUND LAKE,IL,60073-2390,ZZ,Not Available,NaN,NaN,47,D58F12-CT,"5/8"" 4X12' FC TYPE X","Drywall (5/8"" Standard White Board)",DRYWALL,Core,10.0,SHT
1,10000434,0,1,2024-02-01,2024-02-01,so,93290,MISC,1528 W REMINGTON LN,NaN,ROUND LAKE,IL,60073-2390,ZZ,Not Available,NaN,NaN,47,D58F12-CT,"5/8"" 4X12' FC TYPE X","Drywall (5/8"" Standard White Board)",DRYWALL,Core,10.0,SHT
2,10000434,0,1,2024-02-01,2024-02-01,so,93290,MISC,1528 W REMINGTON LN,NaN,ROUND LAKE,IL,60073-2390,ZZ,Not Available,NaN,NaN,47,D58F12-CT,"5/8"" 4X12' FC TYPE X","Drywall (5/8"" Standard White Board)",DRYWALL,Core,10.0,SHT
3,10000434,0,1,2024-02-01,2024-02-01,so,93290,MISC,1528 W REMINGTON LN,NaN,ROUND LAKE,IL,60073-2390,ZZ,Not Available,NaN,NaN,47,D58F12-CT,"5/8"" 4X12' FC TYPE X","Drywall (5/8"" Standard White Board)",DRYWALL,Core,10.0,SHT
4,10000434,0,1,2024-02-01,2024-02-01,so,93290,MISC,1528 W REMINGTON LN,NaN,ROUND LAKE,IL,60073-2390,ZZ,Not Available,NaN,NaN,47,D58F12-CT,"5/8"" 4X12' FC TYPE X","Drywall (5/8"" Standard White Board)",DRYWALL,Core,10.0,SHT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564075,949003342,0,12,2024-09-30,2024-10-07,so,306650,55EMONRO,55 E. MONROE ST.,JERRY (773)505-7216,CHICAGO,IL,60603-5713,CN,Commerical New,O,Office,49,FRPLY34CDX,"3/4"" 4X8 F/R TREATED CDX",Lumber - Ply. Fire Treat,LUMBER,Complementary and Other,20.0,SHT
564076,949003342,0,12,2024-09-30,2024-10-07,so,306650,55EMONRO,55 E. MONROE ST.,JERRY (773)505-7216,CHICAGO,IL,60603-5713,CN,Commerical New,O,Office,49,FRPLY34CDX,"3/4"" 4X8 F/R TREATED CDX",Lumber - Ply. Fire Treat,LUMBER,Complementary and Other,20.0,SHT
564077,949003342,0,12,2024-09-30,2024-10-07,so,306650,55EMONRO,55 E. MONROE ST.,JERRY (773)505-7216,CHICAGO,IL,60603-5713,CN,Commerical New,O,Office,49,FRPLY34CDX,"3/4"" 4X8 F/R TREATED CDX",Lumber - Ply. Fire Treat,LUMBER,Complementary and Other,20.0,SHT
564078,949003342,0,13,2024-09-30,2024-10-07,so,306650,55EMONRO,55 E. MONROE ST.,JERRY (773)505-7216,CHICAGO,IL,60603-5713,CN,Commerical New,O,Office,49,CGAKAM10B,ARM KNURLED ANGLE,Acoustical - Drywall,CEILINGS,Core,100.0,PC


Boolean Filters

In [145]:
filter_df = pd.read_excel('data/Extracted Data.xlsx')
filter_df

,Name,Assigned User,FilterData
0,Gordon and Rockfon Specs IDAHO Job leads,"Jeannette Holvino, Tim Skaggs",(Gordon Industries NEAR Products) OR (Gordon I...
1,*All Products,Leigh Ann Williams,(armstrong NEAR ceiling*) OR (certainteed NEAR...
2,*All Products - Ben,Ben McNeal,"""clark dietrich"" OR ""clarkdietrich"" OR (armstr..."
3,*All Products Bidding,Leigh Ann Williams,(armstrong NEAR ceiling*) OR (certainteed NEAR...
4,*All Products Planning,Leigh Ann Williams,(armstrong NEAR ceiling*) OR (certainteed NEAR...
...,...,...,...
195,All Results_158955,Ryan Hughes,No Filter is available
196,All Results_158998,Chris Bartlett,No Filter is available
197,All Results_159212,Mathew Bryan,No Filter is available
198,All Results_159723,Stu Marshall,No Filter is available


In [146]:
known_filters = filter_df['FilterData'].unique().tolist()
known_filters.remove('No Filter is available')
len(known_filters)

13

In [147]:
known_filters

['(Gordon Industries NEAR Products) OR (Gordon Industries NEAR Manufacturer) OR ("Mullion" NEAR ceiling) OR ("Mullion" NEAR ceilings) OR (Rockfon NEAR ceilings) OR (Rockfon NEAR ceiling)',
 '(armstrong NEAR ceiling*) OR (certainteed NEAR drywall*) OR (certainteed NEAR gypsum*) OR (usg NEAR drywall*) OR (usg NEAR gypsum*) OR ("united states gypsum" NEAR drywall*) OR ("united states gypsum" NEAR gypsum*) OR "national gypsum" OR (eifs NEAR sto) OR eifs OR "g&s" OR "g & s" OR tectum OR "golterman & sabo" OR (turf NEAR "wall panel") OR (turf NEAR "wall panels") OR frasch',
 '"clark dietrich" OR "clarkdietrich" OR (armstrong NEAR ceiling*)',
 '(armstrong NEAR ceiling*) OR (usg NEAR ceiling*) OR ("united states gypsum" NEAR ceiling*) OR (rockfon NEAR ceiling*) OR ("certainteed" NEAR ceiling*)',
 'hammond',
 '(USG NEAR ceiling) OR (USG NEAR ceilings)',
 '(armstrong NEAR ceiling) OR (usg NEAR ceiling) OR ("united states gypsum" NEAR ceiling) OR (armstrong NEAR ceilings) OR (usg NEAR ceilings) O

Boolean text files

In [175]:
boolean_df = pd.read_csv('data/boolean_filters.csv')
boolean_df



,Filter,Query
0,Drywall Sales,(usg NEAR drywall) OR (usg NEAR gypsum) OR ('u...
1,Certainteed Drywall Sales,(certainteed NEAR drywall) OR (certainteed NEA...
2,Georgia Pacific Drywall Sales,('georgia pacific' NEAR drywall) OR ('georgia ...
3,Pabco Drywall Sales,(pabco NEAR drywall) OR (pabco NEAR gypsum)
4,Ceiling Sales,(armstrong NEAR ceiling) OR (armstrong NEAR ce...
5,EIFS & Stucco Sales,(sto NEAR eifs) OR (sto NEAR stucco) OR (stoti...
6,Cultured Stone Sales,(cor NEAR stone) OR (cultured NEAR stone) OR (...
7,Steel Sales,(steel) OR (stud) OR (track) OR (angle) OR ('f...
8,Insulation Sales,(insulation) OR ('mineral wool') OR (fiberglas...
9,Fastener Sales,(fastener) OR (screw) OR (nail) OR (staple) OR...


In [150]:
df_ordered

,Filter,Query
4,Ceiling Sales - Armstrong,(armstrong NEAR ceiling) OR (armstrong NEAR ce...
1,Certainteed Drywall Sales,(certainteed NEAR drywall) OR (certainteed NEA...
0,Drywall Sales,(usg NEAR drywall) OR (usg NEAR gypsum) OR ('u...
2,Georgia Pacific Drywall Sales,('georgia pacific' NEAR drywall) OR ('georgia ...
3,Pabco Drywall Sales,(pabco NEAR drywall) OR (pabco NEAR gypsum)


In [151]:
product_categories = df_ordered.Filter.unique().tolist()
product_categories

['Ceiling Sales - Armstrong',
 'Certainteed Drywall Sales',
 'Drywall Sales',
 'Georgia Pacific Drywall Sales',
 'Pabco Drywall Sales']

Construct Connect data

In [152]:
construct_connect_data = pd.read_pickle('data/construct_connect_data.pkl')


In [ ]:
cc_data_Subset = construct_connect_data[:5]
cc_projects_json_records = cc_data_Subset.to_json(orient='records')


In [177]:
cc_projects_json_records

'[{"ProjectID":1006882884,"DataSourceID":"US","Title":"Bojangles \\/ Chicago","Stage":"Pre-Design","URL":"http:\\/\\/insight.cmdgroup.com\\/SingleSignOn\\/ProjectDetails\\/1006882884\\/1\\/","UpdateDate":"2024-02-16","IsProspective":false,"UpdateText":"Project Details or Scope was Added\\/Updated","Valuation_Value":"900000.00","Valuation_Currency":"USD","Valuation_ValueType":"Staff Estimate Value","Parameters_Parameter_Ownership":"Private","Parameters_Parameter_WorkType":"New","Parameters_Parameter_CommenceDate":"2025-09-05","Parameters_Parameter_FloorArea":null,"Parameters_Parameter_Structures":1.0,"Parameters_Parameter_Units":null,"Parameters_Parameter_BidDate":null,"Parameters_Parameter_FloorAreaUnitofMeasure":null,"Parameters_Parameter_IsSingleTrade":false,"DocumentAvailability_Plans":false,"DocumentAvailability_Specs":false,"DocumentAvailability_Addenda":false,"ParentCategories_PrimaryCategoryName":"Restaurants","ParentCategories_ParentCategory":[{"_Name":null,"ns0:SubCategories":

LLM model

In [154]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)



In [155]:
MODEL_ID = "gemini-1.5-flash" 

model = GenerativeModel(
    MODEL_ID,
    safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },

)


Justin's prompt

In [156]:

question = f'''

**Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low and not relevant.

**Product Categories:**

{product_categories}

**Instructions:**

1. **Analyze the provided JSON data** representing ConstructConnect projects to understand relevant fields and data.
2. **Utilize the Boolean filters** corresponding to each product category (provided below) to identify relevant keywords and phrases.
3. **Classify projects:**
    * In stages indicating active procurement (e.g., "Biddate Set", "SUBBIDS: ASAP", "Post Bid", "Low Bids Announced").
    * Containing a substantial number of matching products and phrases across multiple fields, prioritizing specific materials mentioned.
    * Near the specified geographical area.
    * If all else is equal, assign higher priority to higher value projects.
4. **Estimate the relevancy of each project** based on the strength of product matches, project stage, proximity to specified geography, and total dollar amount. 
5. **Respond with a list of project evaluations** in valid JSON as provided in the Example Output below. Classify the projects as very high, high, moderate, low, very low and not relevant.


**JSON Project Data:** 
{cc_projects_json_records}

**Geographical Area:**
Any

**Boolean Filters:**
{f_dict}

**Example Output:**
[
{{"ProjectID": 1006193703, "Relevance Classification": High, "Reasoning": reason for match}},
{{"ProjectID": 1006193700, "Relevance Classification": Low, "Reasoning": reason for match}}, 
{{"ProjectID": 1006193702, "Relevance Classification": Not relevant, "Reasoning": reason for match}}
]

'''

prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")


Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "Relevance Classification": "Not relevant",
    "Reasoning": "Project is in the pre-design stage, no mention of drywall or ceilings, and is a restaurant project."
  },
  {
    "ProjectID": 1006877272,
    "Relevance Classification": "Very low",
    "Reasoning": "Project is in the General Contractor Award stage, no mention of drywall or ceilings, but the project is a multi-residential development, indicating potential for drywall use."
  },
  {
    "ProjectID": 1006887109,
    "Relevance Classification": "High",
    "Reasoning": "Project is in the Construction Underway stage and is a mixed-use development with 149 units, indicating a high potential for drywall use.  The project is in the specified geographic area."
  },
  {
    "ProjectID": 1006894126,
    "Relevance Classification": "Moderate",
    "Reasoning": "Project is in the Design Development stage, no mention of drywall or ceilings, but the project is a mixed-use developmen

Variation 1

In [157]:
question = f''' You are a system used for classifying ConstructConnect projects based on relevancy. 
Your task is to classify the provided project as high, medium and low relevance based on the probablity of the project to win a bid. 
The projects are classified based on the categories of the products, the project stage, the geographical area, and the total dollar amount.
You are provided with:
1.  a JSON data representing ConstructConnect projects {cc_projects_json_records},
2. a list of product categories{product_categories} of interest, 
3. boolean filters corresponding to each product category {f_dict}.
You need to analyze the JSON data to understand the relevant fields and data, utilize the boolean filters to identify relevant keywords and phrases, and classify the projects based on the provided instructions.
You need to respond with a list of project evaluations in valid JSON format, including the ProjectID, Relevance Classification, and Reasoning for the classification.
The Relevance Classification should be High, Medium, or Low, indicating the probability of the project to win a bid.
The Reasoning should provide a brief explanation of the classification, including the strength of product matches, project stage, proximity to the specified geography, and total dollar amount.'''


prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")



Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "RelevanceClassification": "Low",
    "Reasoning": "The project is in the 'Pre-Design' stage, indicating that it is very early in the planning process. Additionally, the project scope involves 'Site work and new construction of a restaurant' which does not strongly align with the provided product categories.  The project valuation of $900,000 is relatively small."
  },
  {
    "ProjectID": 1006877272,
    "RelevanceClassification": "Medium",
    "Reasoning": "The project is in the 'General Contractor Award' stage, suggesting that the general contractor has been selected, and subcontractor bidding may be occurring. While the project category 'Apartments' does not directly match the provided product categories, the project scope includes 'Demolition' and 'New Construction', which could involve drywall work. The project valuation of $200,000 is moderate."
  },
  {
    "ProjectID": 1006887109,
    "RelevanceClassification": "Low",
   

Justin's prompt - not considering location.
Variation 2

In [158]:
question = f''' You are an expert system used for classifying ConstructConnect projects based on relevance. 
Your task is to classify the provided project as very high, high, moderate, low, very low and not relevant to determine the likelihood of conversion to sales. 

**Instructions:**

1. **Analyze the provided JSON data** representing ConstructConnect projects to understand relevant fields and data.
2. **Utilize the Boolean filters** corresponding to each product category (provided below) to identify relevant keywords and phrases. 
A project is considered a match for a category if *any* of the keywords within the category's filter are found in the project data. 
Multiple occurrences of a keyword within a field should be counted as a single match. Case-insensitive matching should be performed.
3. **Classify projects:**
    * In stages indicating active procurement (e.g., "Biddate Set", "SUBBIDS: ASAP", "Post Bid", "Low Bids Announced").
    * Near the specified geographical area.
    * If all else is equal, assign higher priority to higher value projects.
4. **Estimate the relevancy of each project** based on the strength of product matches, project stage, proximity to specified geography, and total dollar amount. 
5. **Respond with a list of project evaluations** with an in-depth reasoning for relevancy rating (products, product categories, location, value etc.) in a valid JSON as provided in the Example Output below. Classify the projects as very high, high, moderate, low, very low and not relevant.


**JSON Project Data:** 
{cc_projects_json_records}

**Product Categories:**
{product_categories}

**Geographical Area:**
Chicago, IL

**Boolean Filters:**
{f_dict}


*Example Output:**
[
{{"ProjectID": 1006193703, "Relevance Classification": High, "Reasoning": "Product", "Product categories", "location", "value"}},
{{"ProjectID": 1006193700, "Relevance Classification": Low, "Reasoning": "Product", "Product categories", "location", "value"}}, 
{{"ProjectID": 1006193702, "Relevance Classification": Medium, "Reasoning": "Product", "Product categories", "location", "value"}}
]

'''

prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")



Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "Relevance Classification": "Very Low",
    "Reasoning": "The project is in the 'Pre-Design' stage, which is too early for any product consideration. Also, the project is valued at $900,000.00, which is considered low for our products."
  },
  {
    "ProjectID": 1006877272,
    "Relevance Classification": "Very Low",
    "Reasoning": "While the project is in the 'General Contractor Award' stage, indicating active procurement, the project value of $200,000.00 is too low for our products."
  },
  {
    "ProjectID": 1006887109,
    "Relevance Classification": "Moderate",
    "Reasoning": "The project is in the 'Construction Underway' stage, which is past the initial bid phase. However, it's a large project with a value of $41,000,000.00, making it moderately relevant as it may have future opportunities for our products. The project is in Chicago, IL, within the specified geographical area."
  },
  {
    "ProjectID": 1006894126,
    "

Variation 3- weighted prompt

In [159]:
question = f''' You are an expert system used for classifying ConstructConnect projects based on relevance. 
Your task is to classify the provided project as very high, high, moderate, low, very low and not relevant to determine the likelihood of conversion to sales. 

**Instructions:**

1. **Analyze the provided JSON data** representing ConstructConnect projects to understand relevant fields and data.
2. **Utilize the Boolean filters** corresponding to each product category (provided below) to identify relevant keywords and phrases. 
A project is considered a match for a category if *any* of the keywords within the category's filter are found in the project data. 
Multiple occurrences of a keyword within a field should be counted as a single match. Case-insensitive matching should be performed.
3. **Classify projects:**
    * In stages indicating active procurement (e.g., "Biddate Set", "SUBBIDS: ASAP", "Post Bid", "Low Bids Announced").
    * Near the specified geographical area.
    * If all else is equal, assign higher priority to higher value projects.
4. Using the **Relevance Score Calculation** calculate the score for each criteria- product matches, project stage, proximity to specified geography, and total dollar amount. 
5. Calculate **Total Relevance Score** = Sum of all criteria points. 
5. Using the **Relevance Classification** classify the projects as very high, high, moderate, low, very low and not relevant.
6. Respond with a list of project evaluations** with an in-depth reasoning for relevancy rating (products, product categories, location, value etc.) in a valid JSON as provided in the Example Output below. Classify the projects as very high, high, moderate, low, very low and not relevant.


**JSON Project Data:** 
{cc_projects_json_records}

**Product Categories:**
{product_categories}

**Geographical Area:**
Chicago, IL

**Boolean Filters:**
{f_dict}

**Relevance Score Calculation:**
*   Project Stage points: Bid Date Set: 5, Cancelled: 1, Construction Documents: 4, Construction Underway: 3, Design Development: 3, Disqualified Lead: 2, Duplicate Project: 0, General Contractor Award: 4, Low Bids Announced: 4, Post Bid: 3, Pre-Design: 1, Schematic Design: 2, SUBBIDS: ASAP: 5 
*   Product Matches points: High (5 points - substantial match), Medium (3 points - some matches), Low (1 point - few or no matches)
*   Proximity points: High (5 points - if the project is in the same state as **Geographical Area**), Medium (2 points - neighboring states), Low (1 point - other states)
*   Project Value points: Very High (5 points - above 4M USD), High (3 point - between 3M USD and 4M USD), Medium (2 points - between 2M USD and 3M USD), Low (1 point - below 2M USD)

**Total Relevance Score = Sum of all criteria points.

**Relevance Classification:**
* very high: Total Score >= 20
* high: Total Score >= 15 and < 20
* moderate: Total Score >= 10 and < 15
* low: Total Score >= 5 and < 10
* very low: Total Score < 5
* not relevant: Total Score = 0

*Example Output:**
[
{{"ProjectID": 1006193703, "Relevance Classification": High, "Reasoning": "Project Stage points, Product Matches points, Proximity points, Project Value points, total points"}},
{{"ProjectID": 1006193700, "Relevance Classification": Low, "Reasoning": "Project Stage points, Product Matches points, Proximity points, Project Value points, total points"}}, 
{{"ProjectID": 1006193702, "Relevance Classification": Medium, "Reasoning": "Project Stage points, Product Matches points, Proximity points, Project Value points, total points"}}
]

'''

prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")



Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "Relevance Classification": "Very Low",
    "Reasoning": "The project is in the 'Pre-Design' stage, which has a low relevance score of 1. The project does not match any of the product categories, resulting in 0 product match points. The project is located in Chicago, IL, which is the specified geographical area, giving it a high proximity score of 5. The project has a valuation of 900,000 USD, which is below 2M USD, resulting in a low project value score of 1.  The total relevance score is 7, classifying the project as 'Very Low' relevance."
  },
  {
    "ProjectID": 1006877272,
    "Relevance Classification": "Low",
    "Reasoning": "The project is in the 'General Contractor Award' stage, which has a high relevance score of 4. The project does not match any of the product categories, resulting in 0 product match points. The project is located in Chicago, IL, which is the specified geographical area, giving it a high proximity sco

Variation 4- Passing specific "Stage" list with wieghted prompt

In [160]:
stage_filter = {"high": ["Biddate Set", "Construction Documents", "General Contractor Award", "Low Bids Announced", "SUBBIDS: ASAP"], "moderate": ["Construction Underway", "Post Bid"], "ignore": ["Cancelled", "Design Development", "Disqualified Lead", "Duplicate Project", "Pre-Design", "Schematic Design"]}

In [161]:
## Gemini Prompt for ConstructConnect Project Classification
question = f'''
## ConstructConnect Project Classification

**Objective:** Classify ConstructConnect projects based on their likelihood of conversion to sales, providing a structured JSON output with relevance classification and detailed reasoning.

**Input:**

1. **Project Data (JSON):** A JSON object containing project information.  {cc_projects_json_records}

2. **Product Categories of Interest :** A list of strings representing the product categories. {product_categories}

3. **Boolean Filters:** A JSON object representing the boolean filters. {f_dict}

4. **Geographical Area of Interest :**  Chicago, IL, USA

5. **Target Project Stages:** A JSON object defining high, moderate and ignore stages. {stage_filter}


**Instructions:**

Analyze the provided project data and assign a relevancy rating (Very High, High, Moderate, Low, Very Low, Not Relevant) based on the criteria outlined below.  Provide a structured JSON output as shown in the example.

**Relevancy Criteria:**

* **Product Fit (Max 5 points):**
    * Project involves a product category of interest AND matches boolean filters: 5 points per category
    * Project involves a product category of interest but does NOT match boolean filters: 2 points per category
    * No relevant product categories: 0 points
* **Geographic Relevance (Max 3 points):**
    * Project's address StateProvince is within **Geographical Area of Interest**: 3 points
    * Project's address StateProvince is borders the **Geographical Area of Interest**: 1 point
    * Project's address StateProvince is outside and does not border the **Geographical Area of Interest**: 0 points
* **Project Stage (Max 5 points):**
    * High priority stage: 5 points
    * Moderate priority stage: 3 points
    * Stage to ignore: 0 points
* **Valuation (Max 3 points):**
    * High valuation (more than $5,000,000): 3 points
    * Medium valuation (between $3,000,000 to $5,000,000 USD): 2 points
    * Low valuation (between $1,000,000 to $3,000,000 ): 1 point
    * No valuation (lass than $1,000,000): 0 point


**Output:**

Return only a JSON object with the following structure:
```json
[{{
  "ProjectID": 1006193703,
  "Relevance Classification": "<relevancy_rating>",
  "Reasoning": "<detailed_reasoning_string>",
  "Product Fit Points": <product_fit_points>,
  "Geographic Relevance Points": <geographic_relevance_points>,
  "Project Stage Points": <project_stage_points>,
  "Valuation Points": <valuation_points>,
  "Total Points": <total_points>
}},
{{
  "ProjectID": 1006194567,
  "Relevance Classification": "<relevancy_rating>",
  "Reasoning": "<detailed_reasoning_string>",
  "Product Fit Points": <product_fit_points> ,
  "Geographic Relevance Points": <geographic_relevance_points> ,
  "Project Stage Points": <project_stage_points>,
  "Valuation Points": <valuation_points>,
  "Total Points": <total_points>
}},
...]
```

'''

prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")



Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "Relevance Classification": "Not Relevant",
    "Reasoning": "The project is in the 'Pre-Design' stage, which is ignored. It also does not involve any product categories of interest.",
    "Product Fit Points": 0,
    "Geographic Relevance Points": 3,
    "Project Stage Points": 0,
    "Valuation Points": 1,
    "Total Points": 4
  },
  {
    "ProjectID": 1006877272,
    "Relevance Classification": "Not Relevant",
    "Reasoning": "The project is in the 'General Contractor Award' stage, which is a high priority. However, it does not involve any product categories of interest.",
    "Product Fit Points": 0,
    "Geographic Relevance Points": 3,
    "Project Stage Points": 5,
    "Valuation Points": 0,
    "Total Points": 8
  },
  {
    "ProjectID": 1006887109,
    "Relevance Classification": "Not Relevant",
    "Reasoning": "The project is in the 'Construction Underway' stage, which is moderate priority. However, it does not involv

In [ ]:
# Columns to print
columns_to_print = ['ProjectID', 'Stage', 'Valuation_Value', 'ParentCategories_PrimaryCategoryName', 'ParentCategories_ParentCategory', 'Addresses_Address']

# Iterating through rows and printing selected columns
for index, row in cc_data_Subset.iterrows():
    values = [row[col] for col in columns_to_print]
    print(f"Row {index}: {', '.join(map(str, values))}")

Variation 5- Pre-filtering projects based on location instead of including in the prompt. 

In [162]:
first_row_address = construct_connect_data.iloc[0]['Addresses_Address']
print(first_row_address)

[{'_ProjectAddressType': None, 'ns0:AddressLine1': '681 Springfield St', 'ns0:AddressLine2': None, 'ns0:City': 'Agawam', 'ns0:CountryRegion': 'UNITED STATES', 'ns0:County': 'Hampden', 'ns0:Latitude': Decimal('42.072375000'), 'ns0:Longitude': Decimal('-72.675163000'), 'ns0:StateProvince': 'MA', 'ns0:ZipPostalCode': '01030'}]


In [163]:
city_name = "Chicago"
state_abbr = "IL"
zip_code_list = get_zip_codes(city_name, state_abbr)

In [167]:
location_specific_cc_data = construct_connect_data[construct_connect_data['Addresses_Address'].apply(lambda x: any(addr.get('ns0:ZipPostalCode') in zip_code_list for addr in x))]
location_specific_cc_data

,ProjectID,DataSourceID,Title,Stage,URL,UpdateDate,IsProspective,UpdateText,Valuation_Value,Valuation_Currency,Valuation_ValueType,Parameters_Parameter_Ownership,Parameters_Parameter_WorkType,Parameters_Parameter_CommenceDate,Parameters_Parameter_FloorArea,Parameters_Parameter_Structures,Parameters_Parameter_Units,Parameters_Parameter_BidDate,Parameters_Parameter_FloorAreaUnitofMeasure,Parameters_Parameter_IsSingleTrade,DocumentAvailability_Plans,DocumentAvailability_Specs,DocumentAvailability_Addenda,ParentCategories_PrimaryCategoryName,ParentCategories_ParentCategory,Addresses_Address,Contracts_Contract,ProjectEvents_ProjectEvent,Companies,Users,UpdateSummarizations_UpdateSummary,PlanSpecs,Details_Detail_Scope,Details_Detail_Notes,Details_Detail_Details,Details_Detail,Materials_Material,RSMeansMaterialDivisions_Division_Concrete,RSMeansMaterialDivisions_Division_Masonry,RSMeansMaterialDivisions_Division_Metals,RSMeansMaterialDivisions_Division_WoodPlasticsandComposites,RSMeansMaterialDivisions_Division_ThermalandMoistureProtection,RSMeansMaterialDivisions_Division_Openings,RSMeansMaterialDivisions_Division_Finishes,RSMeansMaterialDivisions_Division_Furnishings,RSMeansMaterialDivisions_Division_ConveyingEquipment,RSMeansMaterialDivisions_Division_FireSuppression,RSMeansMaterialDivisions_Division_Plumbing,RSMeansMaterialDivisions_Division_HeatingVentilatingandAirConditioningHVAC,RSMeansMaterialDivisions_Division_Electrical,RSMeansMaterialDivisions_Division_Communications,RSMeansMaterialDivisions_Division_ElectronicSafetyandSecurity,RSMeansMaterialDivisions_Division_Earthwork,RSMeansMaterialDivisions_Division_Utilities,Notes,ListDate,Connections,Parameters_Parameter_CompletionDate,Parameters_Parameter_MandatoryPreBidConferenceDate,RSMeansMaterialDivisions_Division_Specialties,RSMeansMaterialDivisions_Division_Equipment,Parameters_Parameter_FloorsAboveGround,RSMeansMaterialDivisions_Division_GeneralRequirements,RSMeansMaterialDivisions_Division_SpecialConstruction,Parameters_Parameter_LEEDCertificationIntent,Parameters_Parameter_ParkingSpaces,Parameters_Parameter_BidTime,Parameters_Parameter_FloorsBelowGround,ProjectParentID,Parameters_Parameter_SiteArea,Parameters_Parameter_SiteAreaUnitofMeasure,Parameters_Parameter_GreenBuildingCertification,Details_Detail_ContractConditions,Details,Details_Detail_Subbids,Details_Detail_Quantity_Unitprices,Parameters_Parameter_SingleTradeClassification,timeCreated,sourceFile,Notes_Note,Details_Detail_Status
1120,1006882884,US,Bojangles / Chicago,Pre-Design,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-02-16,False,Project Details or Scope was Added/Updated,900000.00,USD,Staff Estimate Value,Private,New,2025-09-05,NaN,1.0,NaN,None,None,False,False,False,False,Restaurants,"[{'_Name': None, 'ns0:SubCategories': {'ns0:Su...","[{'_ProjectAddressType': None, 'ns0:AddressLin...",[],"[{'ns0:Event': 'Start Date', 'ns0:EventDate': ...","[{'ns0:Company': [{'_BiddingRole': None, '_Com...",[{'ns0:User': [{'_ProjectFlaggedForExport': No...,"[{'_Summary': None, '_UpdateSummaryDate': None...",[],[Site work and new construction of a restauran...,"[Development include(s): New Construction, Si...",[],"[{'_': None, '_DetailType': None}, {'_': None,...",[],"[{'_': None, '_Code': None, '_InstallationCost...",[],"[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...",[],[],"[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...",[],"[{'_': None, '_Code': None, '_InstallationCost...","[{'_': None, '_Code': None, '_InstallationCost...",[],<NA>,2023-08-16,[],None,None,"[{'_': None, '_Code': None, '_InstallationCost...",[],NaN,[],[],None,NaN,None,NaN,NaN,NaN,None,

In [168]:
location_specific_subset = location_specific_cc_data[:5]
cc_projects_json_records = location_specific_subset.to_json(orient='records')


In [169]:
## Gemini Prompt for ConstructConnect Project Classification
question = f'''
## ConstructConnect Project Classification

**Objective:** Classify ConstructConnect projects based on their likelihood of conversion to sales, providing a structured JSON output with relevance classification and detailed reasoning.

**Input:**

1. **Project Data (JSON):** A JSON object containing project information.  {cc_projects_json_records}

2. **Product Categories of Interest :** A list of strings representing the product categories. {product_categories}

3. **Boolean Filters:** A JSON object representing the boolean filters. {f_dict}

4. **Target Project Stages:** A JSON object defining high, moderate and ignore stages. {stage_filter}


**Instructions:**

Analyze the provided project data and assign a relevancy rating (Very High, High, Moderate, Low, Very Low, Not Relevant) based on the criteria outlined below.  Provide a structured JSON output as shown in the example.

**Relevancy Criteria:**

* **Product Fit (Max 5 points):**
    * Project involves a product category of interest AND matches boolean filters: 5 points per category
    * Project involves a product category of interest but does NOT match boolean filters: 2 points per category
    * No relevant product categories: 0 points
* **Project Stage (Max 5 points):**
    * High priority stage: 5 points
    * Moderate priority stage: 3 points
    * Stage to ignore: 0 points
* **Valuation (Max 3 points):**
    * High valuation (more than $5,000,000): 3 points
    * Medium valuation (between $3,000,000 to $5,000,000 USD): 2 points
    * Low valuation (between $1,000,000 to $3,000,000 ): 1 point
    * No valuation (lass than $1,000,000): 0 point


**Output:**

Return only a JSON object with the following structure:
```json
[{{
  "ProjectID": 1006193703,
  "Relevance Classification": "<relevancy_rating>",
  "Reasoning": "<detailed_reasoning_string>",
  "Product Fit Points": <product_fit_points>,
  "Project Stage Points": <project_stage_points>,
  "Valuation Points": <valuation_points>,
  "Total Points": <total_points>
}},
{{
  "ProjectID": 1006194567,
  "Relevance Classification": "<relevancy_rating>",
  "Reasoning": "<detailed_reasoning_string>",
  "Product Fit Points": <product_fit_points> ,
  "Project Stage Points": <project_stage_points>,
  "Valuation Points": <valuation_points>,
  "Total Points": <total_points>
}},
...]
```

'''

prompt = question
contents = [prompt]

# Generate text using non-streaming method
response = model.generate_content(contents)

# Print generated text and usage metadata
print(f"\nAnswer:\n{response.text}")



Answer:
```json
[
  {
    "ProjectID": 1006882884,
    "Relevance Classification": "Not Relevant",
    "Reasoning": "The project is in the 'Pre-Design' stage, which is categorized as 'ignore'. The project involves the 'Restaurants' category, which is not a product category of interest. The valuation is 900000.00 USD, which is low but does not contribute to relevancy because the stage is ignored.",
    "Product Fit Points": 0,
    "Project Stage Points": 0,
    "Valuation Points": 1,
    "Total Points": 1
  },
  {
    "ProjectID": 1006877272,
    "Relevance Classification": "High",
    "Reasoning": "The project is in the 'General Contractor Award' stage, which is categorized as 'high'. The project involves the 'Apartments' category, which is not a product category of interest. The valuation is 200000.00 USD, which is low but does not negatively impact the relevancy because the stage is high.",
    "Product Fit Points": 0,
    "Project Stage Points": 5,
    "Valuation Points": 1,
    "T

In [170]:
# Columns to print
columns_to_print = ['ProjectID', 'Stage', 'Valuation_Value', 'ParentCategories_PrimaryCategoryName', 'ParentCategories_ParentCategory', 'Addresses_Address']

# Iterating through rows and printing selected columns
for index, row in location_specific_subset.iterrows():
    values = [row[col] for col in columns_to_print]
    print(f"Row {index}: {', '.join(map(str, values))}")

Row 1120: 1006882884, Pre-Design, 900000.00, Restaurants, [{'_Name': None, 'ns0:SubCategories': {'ns0:SubCategory': array(['Restaurants'], dtype=object)}}], [{'_ProjectAddressType': None, 'ns0:AddressLine1': 'To Be Determined', 'ns0:AddressLine2': None, 'ns0:City': 'Chicago', 'ns0:CountryRegion': 'UNITED STATES', 'ns0:County': 'Cook', 'ns0:Latitude': Decimal('41.875734000'), 'ns0:Longitude': Decimal('-87.625562000'), 'ns0:StateProvince': 'IL', 'ns0:ZipPostalCode': '60604'}]
Row 1121: 1006877272, General Contractor Award, 200000.00, Apartments, [{'_Name': None, 'ns0:SubCategories': {'ns0:SubCategory': array(['Apartments'], dtype=object)}}], [{'_ProjectAddressType': None, 'ns0:AddressLine1': '1952 W 21st St', 'ns0:AddressLine2': None, 'ns0:City': 'Chicago', 'ns0:CountryRegion': 'UNITED STATES', 'ns0:County': 'Cook', 'ns0:Latitude': Decimal('41.849103000'), 'ns0:Longitude': Decimal('-87.667335000'), 'ns0:StateProvince': 'IL', 'ns0:ZipPostalCode': '60608'}]
Row 1122: 1006887109, Constructi